# Conditional VAE for Antimicrobial Peptide Generation

**Combines:**
- Baseline cVAE with APD6 data (5984 sequences, 7 binary conditions)
- Dean & Walper (ACS Omega 2020): LSTM architecture, character dropout, KL annealing
- Bowman et al. (2015): word dropout to prevent posterior collapse
- Latent interpolation & physicochemical property analysis

**Architecture:** LSTM encoder → 32D latent → LSTM decoder with condition injection

## 1. Data Loading

In [ ]:
import numpy as np
import pandas as pd
import pickle
import math
import random

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from pathlib import Path
from tqdm.auto import tqdm, trange

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if torch.backends.mps.is_available():
    torch.mps.manual_seed(SEED)

In [ ]:
# Load preprocessed APD6 data
DATA_PATH = Path("data/preprocessed/data.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"{DATA_PATH} not found — run data/notebooks/preprocessing.ipynb first")

df = pd.read_csv(DATA_PATH)

seq_col = 'Sequence'
len_col = 'Length'
condition_cols = [
    'is_antibacterial', 'is_anti_gram_positive', 'is_anti_gram_negative',
    'is_antifungal', 'is_antiviral', 'is_antiparasitic', 'is_anticancer'
]

existing_cols = [col for col in ['APD ID', seq_col, len_col] + condition_cols if col in df.columns]
df = df[existing_cols].dropna(subset=[seq_col])
for col in condition_cols:
    if col in df.columns:
        df[col] = df[col].astype(int)

print(f"Dataset size: {df.shape}")
print(f"\nCondition distribution:")
for col in condition_cols:
    print(f"  {col}: {df[col].sum()} positive ({df[col].mean()*100:.1f}%)")
df.head()

In [ ]:
# Build vocabulary from sequences
all_chars = set()
for seq in df[seq_col]:
    all_chars.update(seq)

special_tokens = ['<PAD>', '<SOS>', '<EOS>', '<UNK>']
vocab_list = special_tokens + sorted(all_chars)
char2idx = {ch: i for i, ch in enumerate(vocab_list)}
idx2char = {i: ch for ch, i in char2idx.items()}
vocab_size = len(vocab_list)

PAD_IDX = char2idx['<PAD>']
SOS_IDX = char2idx['<SOS>']
EOS_IDX = char2idx['<EOS>']
UNK_IDX = char2idx['<UNK>']

print(f"Vocabulary size: {vocab_size}")
print(f"Amino acids: {sorted(all_chars)}")

In [ ]:
def tokenize(seq, char2idx, max_len, add_sos_eos=True):
    """Convert sequence string to padded token indices."""
    tokens = [char2idx.get(ch, UNK_IDX) for ch in seq]
    if add_sos_eos:
        tokens = [SOS_IDX] + tokens + [EOS_IDX]
    if len(tokens) < max_len:
        tokens += [PAD_IDX] * (max_len - len(tokens))
    else:
        tokens = tokens[:max_len]
    return tokens

# Max length with SOS/EOS
max_len = df[len_col].max() + 2
print(f"Max sequence length (with SOS/EOS): {max_len}")

# Tokenize all sequences
tokenized_seqs = [tokenize(seq, char2idx, max_len) for seq in df[seq_col]]
conditions = df[condition_cols].values.astype(np.float32)
real_lengths = np.array([min(len(seq) + 2, max_len) for seq in df[seq_col]])

In [ ]:
class AMPDataset(Dataset):
    """Dataset for antimicrobial peptide sequences with multi-hot conditions."""
    def __init__(self, sequences, lengths, conditions):
        self.sequences = torch.LongTensor(sequences)
        self.lengths = torch.LongTensor(lengths)
        self.conditions = torch.FloatTensor(conditions)  # multi-hot: peptide can have multiple activities

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return self.sequences[idx], self.lengths[idx], self.conditions[idx]

dataset = AMPDataset(tokenized_seqs, real_lengths, conditions)

# Train/val/test split (80/10/10)
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(
    dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

batch_size = 256
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

## 2. Model Architecture

**Anti-collapse mitigations** (He et al. 2019; Lucas et al. 2019):
1. **Reduced decoder** (512 hidden) with stronger encoder (1024 hidden) — asymmetric power balance forces decoder to rely on z
2. **Free bits** (Kingma et al. 2016): per-dimension KL floor of 0.25 nats — mathematically prevents KL→0
3. **Cyclical KL annealing** (Fu et al. 2019): beta cycles 0→1 four times — periodically re-opens the information channel
4. **Lagging inference** (He et al. 2019): pre-train encoder for several steps before each decoder update in early epochs — gives encoder a head start
5. **z-injection at every timestep**: z+cond concatenated to decoder input at each step, not just init — decoder *must* attend to z throughout generation
6. **Character dropout** (Bowman et al. 2015): 30% rate (reduced from 50% to avoid routing-around)

In [ ]:
# =====================================================================
# Hyperparameters (with anti-collapse mitigations)
# =====================================================================
embed_dim = 128             # Character embedding dimension
enc_hidden_dim = 1024       # Encoder LSTM hidden (strong encoder)
dec_hidden_dim = 512        # Decoder LSTM hidden (weaker decoder — forces z reliance)
latent_dim = 32             # Latent space dimension
cond_dim = len(condition_cols)  # 7 binary conditions
num_layers = 1              # Single LSTM layer
dropout = 0.2               # Embedding/output dropout
word_dropout_rate = 0.3     # Character dropout (reduced from 0.5 to prevent routing-around)
lr = 1e-3

# Anti-collapse settings
free_bits = 0.25            # Min KL per latent dim (nats) — prevents KL→0
cyclical_period = 10        # Epochs per beta cycle (0→1)
n_cycles = 4                # Number of full beta cycles
lagging_epochs = 10         # Epochs to use lagging inference
lagging_inner_steps = 5     # Encoder-only steps per decoder step during lagging

if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

print(f"Device: {device}")
print(f"\nHyperparameters:")
print(f"  enc_hidden={enc_hidden_dim}, dec_hidden={dec_hidden_dim}, latent={latent_dim}")
print(f"  word_dropout={word_dropout_rate}, free_bits={free_bits}")
print(f"  cyclical: {n_cycles} cycles x {cyclical_period} epochs")
print(f"  lagging inference: {lagging_epochs} epochs, {lagging_inner_steps} inner steps")

In [ ]:
class Encoder(nn.Module):
    """
    Strong LSTM Encoder (1024 hidden).
    Asymmetric: encoder is more powerful than decoder to encourage
    putting information into z (He et al. 2019 / Lucas et al. 2019).
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.drop = nn.Dropout(dropout)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, num_layers=num_layers,
                            batch_first=True)
        self.fc_mu = nn.Linear(hidden_dim + cond_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim + cond_dim, latent_dim)

    def forward(self, x, lengths, cond):
        embedded = self.drop(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(
            embedded, lengths.cpu().clamp(min=1), batch_first=True, enforce_sorted=False
        )
        _, (h_n, _) = self.lstm(packed)
        hidden = h_n[-1]
        hidden_cond = torch.cat([hidden, cond], dim=1)
        return self.fc_mu(hidden_cond), self.fc_logvar(hidden_cond)


class Decoder(nn.Module):
    """
    Weaker LSTM Decoder (512 hidden) with z+cond injection at every timestep.

    Two anti-collapse mechanisms:
    1. Smaller hidden size than encoder — can't fully model sequences alone.
    2. z and cond are concatenated to the embedding at EVERY timestep
       (not just used for hidden-state init). The decoder sees z continuously,
       so it learns to depend on it rather than routing around via memory.
    """
    def __init__(self, vocab_size, embed_dim, hidden_dim, latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.latent_dim = latent_dim
        self.cond_dim = cond_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.drop = nn.Dropout(dropout)
        # LSTM input: embedding + z + cond at every step
        self.lstm = nn.LSTM(embed_dim + latent_dim + cond_dim, hidden_dim,
                            num_layers=num_layers, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim, vocab_size)
        # Hidden/cell state init from z+cond (still useful for first step)
        self.init_h = nn.Linear(latent_dim + cond_dim, hidden_dim)
        self.init_c = nn.Linear(latent_dim + cond_dim, hidden_dim)

    def forward(self, x, z, cond, word_dropout_rate=0.0):
        batch_size, seq_len = x.shape
        z_cond = torch.cat([z, cond], dim=1)
        h0 = torch.tanh(self.init_h(z_cond)).unsqueeze(0)
        c0 = torch.tanh(self.init_c(z_cond)).unsqueeze(0)

        # Character dropout
        if self.training and word_dropout_rate > 0:
            mask = (torch.rand(batch_size, seq_len, device=x.device) < word_dropout_rate) & (x != PAD_IDX)
            x = x.clone()
            x[mask] = UNK_IDX

        embedded = self.drop(self.embedding(x))  # (batch, seq_len, embed_dim)

        # Concatenate z and cond to every timestep
        z_expand = z.unsqueeze(1).expand(-1, seq_len, -1)      # (batch, seq_len, latent_dim)
        cond_expand = cond.unsqueeze(1).expand(-1, seq_len, -1) # (batch, seq_len, cond_dim)
        lstm_input = torch.cat([embedded, z_expand, cond_expand], dim=2)

        outputs, _ = self.lstm(lstm_input, (h0, c0))
        return self.fc_out(outputs)


class CVAE(nn.Module):
    """
    Conditional VAE with anti-collapse architecture.
    - Asymmetric encoder (1024) > decoder (512)
    - z injected at every decoder timestep
    """
    def __init__(self, vocab_size, embed_dim, enc_hidden, dec_hidden,
                 latent_dim, cond_dim, dropout=0.2):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, enc_hidden, latent_dim, cond_dim, dropout)
        self.decoder = Decoder(vocab_size, embed_dim, dec_hidden, latent_dim, cond_dim, dropout)
        self.latent_dim = latent_dim
        self.cond_dim = cond_dim

    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + torch.randn_like(std) * std
        return mu

    def forward(self, src, lengths, cond, word_dropout_rate=0.0):
        mu, logvar = self.encoder(src, lengths, cond)
        z = self.reparameterize(mu, logvar)
        logits = self.decoder(src[:, :-1], z, cond, word_dropout_rate)
        return logits, src[:, 1:], mu, logvar


model = CVAE(vocab_size, embed_dim, enc_hidden_dim, dec_hidden_dim,
             latent_dim, cond_dim, dropout).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {n_params:,}")
print(f"\nArchitecture:")
print(model)

## 3. Training

**Anti-collapse training strategy:**
- **Free bits** (Kingma et al. 2016): `KL_dim = max(KL_dim, 0.25)` — each latent dimension must carry at least 0.25 nats
- **Cyclical beta** (Fu et al. 2019): 4 cycles of linear 0→1, each over 10 epochs — re-opens the z channel periodically
- **Lagging inference** (He et al. 2019): during first 10 epochs, train encoder 5 extra steps per batch to give it a head start
- **Gradient clipping** (max norm 5.0) for stability
- **Early stopping** with patience=20

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)


def get_kl_weight(epoch):
    """
    Cyclical KL annealing (Fu et al. 2019).
    Beta linearly ramps 0→1 within each cycle, then resets.
    After all cycles complete, beta stays at 1.
    """
    total_anneal_epochs = n_cycles * cyclical_period
    if epoch >= total_anneal_epochs:
        return 1.0
    position_in_cycle = epoch % cyclical_period
    return position_in_cycle / max(1, cyclical_period - 1)


def loss_function(logits, target, mu, logvar, beta=1.0):
    """
    VAE loss with free bits (Kingma et al. 2016).
    
    Free bits: compute KL per latent dimension, clamp each to a minimum
    of `free_bits` nats. This prevents the encoder from zeroing out any
    dimension — each must carry at least `free_bits` of information.
    """
    recon_loss = criterion(logits.reshape(-1, vocab_size), target.reshape(-1))

    # Per-dimension KL: -0.5 * (1 + logvar - mu^2 - exp(logvar))
    kl_per_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())  # (batch, latent_dim)
    kl_per_dim = kl_per_dim.mean(dim=0)  # Average over batch → (latent_dim,)

    # Free bits: clamp each dimension to minimum
    kl_per_dim = torch.clamp(kl_per_dim, min=free_bits)
    kl_loss = kl_per_dim.sum()  # Sum over latent dims

    return recon_loss + beta * kl_loss, recon_loss, kl_loss


def train_epoch(model, loader, optimizer, epoch, use_lagging=False):
    """
    Train one epoch. If use_lagging=True, applies lagging inference
    (He et al. 2019): train encoder for extra steps before updating decoder.
    """
    model.train()
    total_loss, total_recon, total_kl = 0, 0, 0
    beta = get_kl_weight(epoch)
    pbar = tqdm(loader, desc=f"Train ep{epoch+1}", leave=False)

    for src, lengths, cond in pbar:
        src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)

        # --- Lagging inference: extra encoder-only updates ---
        if use_lagging:
            for _ in range(lagging_inner_steps):
                optimizer.zero_grad()
                mu, logvar = model.encoder(src, lengths, cond)
                z = model.reparameterize(mu, logvar)
                # Decoder is used but its gradients are detached
                with torch.no_grad():
                    logits = model.decoder(src[:, :-1], z, cond, word_dropout_rate)
                target = src[:, 1:]
                # Recompute loss with encoder grads only
                logits_for_enc = model.decoder(src[:, :-1], z, cond, word_dropout_rate)
                loss_enc, _, _ = loss_function(logits_for_enc, target, mu, logvar, beta)
                # Zero decoder grads, keep encoder grads
                loss_enc.backward()
                for p in model.decoder.parameters():
                    if p.grad is not None:
                        p.grad.zero_()
                torch.nn.utils.clip_grad_norm_(model.encoder.parameters(), max_norm=5.0)
                optimizer.step()

        # --- Standard joint update ---
        optimizer.zero_grad()
        logits, target, mu, logvar = model(src, lengths, cond,
                                           word_dropout_rate=word_dropout_rate)
        loss, recon, kl = loss_function(logits, target, mu, logvar, beta)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        total_loss += loss.item()
        total_recon += recon.item()
        total_kl += kl.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}", kl=f"{kl.item():.2f}", beta=f"{beta:.2f}")

    n = len(loader)
    return total_loss/n, total_recon/n, total_kl/n, beta


def eval_epoch(model, loader):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for src, lengths, cond in tqdm(loader, desc="Val", leave=False):
            src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
            logits, target, mu, logvar = model(src, lengths, cond, word_dropout_rate=0.0)
            loss, _, _ = loss_function(logits, target, mu, logvar, beta=1.0)
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# Training loop with all anti-collapse mitigations
num_epochs = 150
best_val_loss = float('inf')
patience = 20  # Increased patience (cyclical annealing causes temporary val spikes)
counter = 0

history = {'train_loss': [], 'val_loss': [], 'recon': [], 'kl': [], 'beta': []}

for epoch in trange(num_epochs, desc="Training"):
    # Lagging inference for first N epochs (He et al. 2019)
    use_lagging = epoch < lagging_epochs

    train_loss, recon, kl, beta = train_epoch(
        model, train_loader, optimizer, epoch, use_lagging=use_lagging
    )
    val_loss = eval_epoch(model, val_loader)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['recon'].append(recon)
    history['kl'].append(kl)
    history['beta'].append(beta)

    if (epoch + 1) % 10 == 0 or epoch == 0:
        lag_str = " [lagging]" if use_lagging else ""
        tqdm.write(f"Epoch {epoch+1:3d}/{num_epochs} | "
                   f"Train: {train_loss:.4f} (R:{recon:.4f} KL:{kl:.4f}) | "
                   f"Val: {val_loss:.4f} | beta={beta:.2f}{lag_str}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        counter = 0
        torch.save(model.state_dict(), 'best_cvae.pt')
    else:
        counter += 1
        if counter >= patience:
            tqdm.write(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(torch.load('best_cvae.pt', weights_only=True))
print(f"\nBest validation loss: {best_val_loss:.4f}")
print(f"Final KL: {history['kl'][-1]:.4f} (should be >> 0 if collapse is fixed)")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train')
axes[0].plot(history['val_loss'], label='Val')
axes[0].set(xlabel='Epoch', ylabel='Loss', title='Total Loss')
axes[0].legend()

axes[1].plot(history['recon'], label='Reconstruction', color='blue')
axes[1].plot(history['kl'], label='KL Divergence', color='orange')
axes[1].set(xlabel='Epoch', ylabel='Loss', title='Loss Components')
axes[1].legend()

axes[2].plot(history['beta'], color='green')
axes[2].set(xlabel='Epoch', ylabel='Beta', title='KL Annealing Schedule')
axes[2].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## 4. Evaluation

Metrics:
- **Validity**: all characters are valid amino acids
- **Uniqueness**: fraction of non-duplicate generated sequences
- **Novelty**: fraction of generated sequences not in training set

In [ ]:
def _decode_from_z_cond(model, z, cond_t, max_gen_len, temperature):
    """Shared autoregressive decoding logic for generation functions.

    Matches the new Decoder architecture: z+cond are concatenated to the
    embedding at every timestep (not just used for hidden-state init).
    """
    z_cond = torch.cat([z, cond_t], dim=1)
    h = torch.tanh(model.decoder.init_h(z_cond)).unsqueeze(0)
    c = torch.tanh(model.decoder.init_c(z_cond)).unsqueeze(0)

    input_token = torch.LongTensor([[SOS_IDX]]).to(device)
    generated = []
    for _ in range(max_gen_len):
        embedded = model.decoder.embedding(input_token)  # (1, 1, embed_dim)
        # Concatenate z and cond at this timestep (matches Decoder.forward)
        lstm_input = torch.cat([embedded, z.unsqueeze(1), cond_t.unsqueeze(1)], dim=2)
        output, (h, c) = model.decoder.lstm(lstm_input, (h, c))
        logits = model.decoder.fc_out(output.squeeze(1))
        probs = torch.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, 1).item()
        if next_token == EOS_IDX:
            break
        generated.append(next_token)
        input_token = torch.LongTensor([[next_token]]).to(device)

    return ''.join([idx2char[idx] for idx in generated
                    if idx not in [PAD_IDX, SOS_IDX, EOS_IDX]])


def generate(model, cond, max_gen_len=None, temperature=1.0, alpha=1.0):
    """
    Generate a peptide sequence from the cVAE.

    Args:
        model: trained CVAE model
        cond: condition vector (list of 7 binary values)
        max_gen_len: maximum generation length (default: max_len - 2)
        temperature: sampling temperature (higher = more diverse)
        alpha: condition strength in [0,1]. At alpha=0, condition is zeroed out
               (unconditional). At alpha=1, full condition.
    """
    if max_gen_len is None:
        max_gen_len = max_len - 2
    model.eval()
    with torch.no_grad():
        cond_t = torch.FloatTensor(cond).unsqueeze(0).to(device) * alpha
        z = torch.randn(1, latent_dim).to(device)
        return _decode_from_z_cond(model, z, cond_t, max_gen_len, temperature)


def generate_from_z(model, z, cond, max_gen_len=None, temperature=1.0, alpha=1.0):
    """Generate from a specific latent point z (for interpolation experiments)."""
    if max_gen_len is None:
        max_gen_len = max_len - 2
    model.eval()
    with torch.no_grad():
        if not isinstance(z, torch.Tensor):
            z = torch.FloatTensor(z).unsqueeze(0).to(device)
        cond_t = torch.FloatTensor(cond).unsqueeze(0).to(device) * alpha
        return _decode_from_z_cond(model, z, cond_t, max_gen_len, temperature)


# Quick test
cond_gram_pos = [0,1,0,0,0,0,0]
print("Sample generations for Gram+:")
for i in range(5):
    print(f"  {i+1}: {generate(model, cond_gram_pos)}")

In [ ]:
def evaluate_generation(model, cond, num_samples=200, train_seqs_set=None, alpha=1.0):
    """Compute Validity, Uniqueness, and Novelty metrics."""
    generated_seqs = [generate(model, cond, alpha=alpha)
                      for _ in trange(num_samples, desc="Generating", leave=False)]

    # Validity: all characters are standard amino acids
    valid_chars = set(all_chars)
    valid_seqs = [s for s in generated_seqs if len(s) > 0 and all(ch in valid_chars for ch in s)]
    validity = len(valid_seqs) / num_samples

    # Uniqueness
    unique_seqs = set(valid_seqs)
    uniqueness = len(unique_seqs) / len(valid_seqs) if valid_seqs else 0

    # Novelty
    if train_seqs_set is not None:
        novel_seqs = [s for s in unique_seqs if s not in train_seqs_set]
        novelty = len(novel_seqs) / len(unique_seqs) if unique_seqs else 0
    else:
        novelty = None

    return validity, uniqueness, novelty, valid_seqs


# Get training sequences for novelty check
train_seqs_raw = [df.iloc[i][seq_col] for i in train_dataset.indices]
train_seqs_set = set(train_seqs_raw)

# Evaluate for different conditions
conditions_to_test = {
    'Antibacterial':    [1,0,0,0,0,0,0],
    'Gram+':            [0,1,0,0,0,0,0],
    'Gram-':            [0,0,1,0,0,0,0],
    'Antifungal':       [0,0,0,1,0,0,0],
    'Antiviral':        [0,0,0,0,1,0,0],
    'Anticancer':       [0,0,0,0,0,0,1],
    'Multi (AB+G+)':    [1,1,0,0,0,0,0],
    'Unconditional':    [0,0,0,0,0,0,0],
}

print(f"{'Condition':<20} {'Validity':>10} {'Uniqueness':>10} {'Novelty':>10}")
print('-' * 55)
for name, cond in tqdm(conditions_to_test.items(), desc="Evaluating conditions"):
    v, u, n, _ = evaluate_generation(model, cond, num_samples=200, train_seqs_set=train_seqs_set)
    print(f"{name:<20} {v:>10.3f} {u:>10.3f} {n:>10.3f}")

## 5. Parametric t-SNE Visualization

In [ ]:
def get_latent_vectors(model, loader):
    """Extract latent means (mu) for all sequences in a dataloader."""
    model.eval()
    latents, conds = [], []
    with torch.no_grad():
        for src, lengths, cond in loader:
            src, lengths, cond = src.to(device), lengths.to(device), cond.to(device)
            mu, _ = model.encoder(src, lengths, cond)
            latents.append(mu.cpu().numpy())
            conds.append(cond.cpu().numpy())
    return np.concatenate(latents), np.concatenate(conds)


train_latents, train_conds = get_latent_vectors(model, train_loader)
print(f"Train latents shape: {train_latents.shape}")

In [ ]:
# Fit standard t-SNE on latent vectors
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30)
train_2d = tsne.fit_transform(train_latents)

# Train parametric t-SNE (small MLP regression from latent -> 2D)
class ParametricTSNE(nn.Module):
    def __init__(self, input_dim, hidden_dim=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 2)
        )
    def forward(self, x):
        return self.net(x)

pt_model = ParametricTSNE(latent_dim).to(device)
pt_optimizer = optim.Adam(pt_model.parameters(), lr=1e-3)
pt_criterion = nn.MSELoss()

X_pt = torch.FloatTensor(train_latents).to(device)
Y_pt = torch.FloatTensor(train_2d).to(device)
dataset_pt = torch.utils.data.TensorDataset(X_pt, Y_pt)
loader_pt = DataLoader(dataset_pt, batch_size=256, shuffle=True)

pt_epochs = 2000
for epoch in trange(pt_epochs, desc="Parametric t-SNE"):
    total_loss = 0
    for xb, yb in loader_pt:
        pt_optimizer.zero_grad()
        loss = pt_criterion(pt_model(xb), yb)
        loss.backward()
        pt_optimizer.step()
        total_loss += loss.item()
    if (epoch+1) % 500 == 0:
        tqdm.write(f"PT-SNE Epoch {epoch+1}/{pt_epochs}, Loss: {total_loss/len(loader_pt):.4f}")

torch.save(pt_model.state_dict(), 'parametric_tsne.pt')

In [ ]:
def project_latents(latents):
    """Project latent vectors to 2D via parametric t-SNE."""
    pt_model.eval()
    with torch.no_grad():
        return pt_model(torch.FloatTensor(latents).to(device)).cpu().numpy()

# Generate sequences for visualization
gen_seqs_gp = [generate(model, cond_gram_pos) for _ in range(100)]
gen_tokens = [tokenize(s, char2idx, max_len) for s in gen_seqs_gp]
gen_lens = [min(len(s)+2, max_len) for s in gen_seqs_gp]
gen_conds = [cond_gram_pos] * len(gen_seqs_gp)
gen_dataset = AMPDataset(gen_tokens, gen_lens, gen_conds)
gen_loader = DataLoader(gen_dataset, batch_size=len(gen_dataset))
gen_latents, _ = get_latent_vectors(model, gen_loader)
gen_2d = project_latents(gen_latents)

# Project training data
train_2d_param = project_latents(train_latents)

# Visualization
plt.figure(figsize=(12, 9))
gram_pos_idx = train_conds[:, 1] == 1
gram_neg_idx = train_conds[:, 2] == 1
antifungal_idx = train_conds[:, 3] == 1
other_idx = ~(gram_pos_idx | gram_neg_idx | antifungal_idx)

plt.scatter(train_2d_param[other_idx, 0], train_2d_param[other_idx, 1],
            c='gray', label='Other', alpha=0.2, s=5)
plt.scatter(train_2d_param[gram_pos_idx, 0], train_2d_param[gram_pos_idx, 1],
            c='green', label='Real Gram+', alpha=0.4, s=10)
plt.scatter(train_2d_param[gram_neg_idx, 0], train_2d_param[gram_neg_idx, 1],
            c='blue', label='Real Gram-', alpha=0.4, s=10)
plt.scatter(train_2d_param[antifungal_idx, 0], train_2d_param[antifungal_idx, 1],
            c='purple', label='Real Antifungal', alpha=0.4, s=10)
plt.scatter(gen_2d[:, 0], gen_2d[:, 1],
            c='red', label='Generated Gram+', s=30, edgecolors='black', zorder=5)

plt.legend(fontsize=10)
plt.title('Parametric t-SNE: Real vs Generated Peptides')
plt.xlabel('t-SNE 1')
plt.ylabel('t-SNE 2')
plt.tight_layout()
plt.show()

## 6. Latent Interpolation (Dean & Walper Concept Vectors)

Following Dean & Walper (2020), we interpolate in latent space between conditions
to generate sequences along a concept vector. This reveals how the model transitions
between different antimicrobial activities.

In [ ]:
def interpolate_conditions(model, cond_start, cond_end, z=None, n_steps=10,
                           temperature=0.8):
    """
    Generate sequences along a condition interpolation path.
    
    Inspired by Dean & Walper's antimicrobial concept vectors:
    interpolate the condition vector from cond_start to cond_end
    while keeping z fixed, to see how the decoded sequence changes.
    
    Args:
        cond_start: starting condition vector
        cond_end: ending condition vector
        z: fixed latent vector (if None, sample from prior)
        n_steps: number of interpolation points
        temperature: sampling temperature
    """
    if z is None:
        z = torch.randn(1, latent_dim).to(device)
    elif not isinstance(z, torch.Tensor):
        z = torch.FloatTensor(z).unsqueeze(0).to(device)
    
    results = []
    alphas = np.linspace(0, 1, n_steps)
    for alpha in alphas:
        # Linear interpolation between conditions
        cond = [(1 - alpha) * s + alpha * e for s, e in zip(cond_start, cond_end)]
        seq = generate_from_z(model, z, cond, temperature=temperature)
        results.append((alpha, cond, seq))
    return results


def interpolate_latent(model, z_start, z_end, cond, n_steps=10, temperature=0.8):
    """
    Interpolate between two points in latent space with fixed condition.
    
    This is the direct analog of Dean & Walper's concept vector interpolation
    (their Figure 3), but in a conditional setting.
    """
    results = []
    alphas = np.linspace(0, 1, n_steps)
    for alpha in alphas:
        z = (1 - alpha) * z_start + alpha * z_end
        seq = generate_from_z(model, z, cond, temperature=temperature)
        results.append((alpha, seq))
    return results


# Example: Interpolate from Gram+ to no condition (concept vector)
print("=" * 70)
print("Condition Interpolation: Gram+ -> Unconditional")
print("=" * 70)
z_fixed = torch.randn(1, latent_dim).to(device)
cond_gp = [0, 1, 0, 0, 0, 0, 0]
cond_none = [0, 0, 0, 0, 0, 0, 0]
results = interpolate_conditions(model, cond_gp, cond_none, z=z_fixed, n_steps=6)
for alpha, cond, seq in results:
    print(f"  alpha={alpha:.2f}  seq={seq}")

print()
print("=" * 70)
print("Condition Interpolation: Gram+ -> Anticancer")
print("=" * 70)
cond_ac = [0, 0, 0, 0, 0, 0, 1]
results = interpolate_conditions(model, cond_gp, cond_ac, z=z_fixed, n_steps=6)
for alpha, cond, seq in results:
    print(f"  alpha={alpha:.2f}  seq={seq}")

In [ ]:
# Visualize interpolation paths in latent space
def plot_interpolation_paths(model, n_paths=5):
    """
    Visualize multiple interpolation paths projected to 2D via parametric t-SNE.
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Background: training data
    ax.scatter(train_2d_param[:, 0], train_2d_param[:, 1],
               c='gray', alpha=0.1, s=3, label='Training data')
    
    cmap = plt.cm.Set1
    for i in range(n_paths):
        z_start = torch.randn(1, latent_dim).to(device)
        z_end = torch.randn(1, latent_dim).to(device)
        
        # Interpolate in latent space
        n_steps = 20
        alphas = np.linspace(0, 1, n_steps)
        z_path = np.array([
            ((1-a) * z_start + a * z_end).cpu().numpy().squeeze()
            for a in alphas
        ])
        
        # Project to 2D
        path_2d = project_latents(z_path)
        color = cmap(i / n_paths)
        ax.plot(path_2d[:, 0], path_2d[:, 1], '-o', color=color,
                markersize=3, linewidth=1.5, alpha=0.7, label=f'Path {i+1}')
        ax.scatter(path_2d[0, 0], path_2d[0, 1], c=[color], s=80, marker='s', zorder=5)
        ax.scatter(path_2d[-1, 0], path_2d[-1, 1], c=[color], s=80, marker='*', zorder=5)
    
    ax.set(xlabel='t-SNE 1', ylabel='t-SNE 2',
           title='Latent Space Interpolation Paths')
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

plot_interpolation_paths(model)

## 7. Physicochemical Property Analysis

Following Dean & Walper (2020), we compute physicochemical properties of generated
peptides to verify they match the expected profiles for each activity class:
- **Net charge** at pH 7 (basic residues K, R, H contribute +1; acidic D, E contribute -1)
- **Hydrophobicity** (Kyte-Doolittle scale mean)
- **Hydrophobic moment** (amphipathicity, assuming alpha-helix with 100 degree rotation)
- **Length distribution**

In [ ]:
# =====================================================================
# Physicochemical Property Calculators
# =====================================================================

# Kyte-Doolittle hydrophobicity scale
KD_SCALE = {
    'A': 1.8, 'C': 2.5, 'D': -3.5, 'E': -3.5, 'F': 2.8,
    'G': -0.4, 'H': -3.2, 'I': 4.5, 'K': -3.9, 'L': 3.8,
    'M': 1.9, 'N': -3.5, 'P': -1.6, 'Q': -3.5, 'R': -4.5,
    'S': -0.8, 'T': -0.7, 'V': 4.2, 'W': -0.9, 'Y': -1.3,
}

# Charge at pH 7: K, R, H are +1; D, E are -1
CHARGE_MAP = {
    'K': 1, 'R': 1, 'H': 0.5,  # H is partially protonated at pH 7 (pKa ~6.0)
    'D': -1, 'E': -1
}


def net_charge(seq):
    """Calculate net charge at pH 7."""
    return sum(CHARGE_MAP.get(aa, 0) for aa in seq)


def hydrophobicity(seq):
    """Mean hydrophobicity (Kyte-Doolittle scale)."""
    if len(seq) == 0:
        return 0
    return np.mean([KD_SCALE.get(aa, 0) for aa in seq])


def hydrophobic_moment(seq, angle=100):
    """
    Hydrophobic moment (amphipathicity) for an alpha-helix.
    
    Uses Eisenberg et al. formulation:
    mu_H = sqrt( (sum H_i sin(i*delta))^2 + (sum H_i cos(i*delta))^2 ) / N
    
    where delta = 100 degrees for alpha-helix (3.6 residues per turn).
    Higher values indicate more amphipathic structure.
    """
    if len(seq) == 0:
        return 0
    delta = math.radians(angle)
    sin_sum = sum(KD_SCALE.get(aa, 0) * math.sin(i * delta) for i, aa in enumerate(seq))
    cos_sum = sum(KD_SCALE.get(aa, 0) * math.cos(i * delta) for i, aa in enumerate(seq))
    return math.sqrt(sin_sum**2 + cos_sum**2) / len(seq)


def compute_properties(sequences):
    """Compute all physicochemical properties for a list of sequences."""
    props = {
        'sequence': sequences,
        'length': [len(s) for s in sequences],
        'net_charge': [net_charge(s) for s in sequences],
        'hydrophobicity': [hydrophobicity(s) for s in sequences],
        'hydrophobic_moment': [hydrophobic_moment(s) for s in sequences],
    }
    return pd.DataFrame(props)


# Test on a known AMP
test_seq = 'GLWSKIKEVGKEAAKAAAKAAGKAALGAVSEAV'
print(f"Test: {test_seq}")
print(f"  Length: {len(test_seq)}")
print(f"  Net charge: {net_charge(test_seq):.1f}")
print(f"  Hydrophobicity: {hydrophobicity(test_seq):.2f}")
print(f"  Hydrophobic moment: {hydrophobic_moment(test_seq):.2f}")

In [ ]:
# Generate sequences for different conditions and compute properties
n_gen = 200
gen_data = {}

conditions_for_analysis = {
    'Gram+':       [0,1,0,0,0,0,0],
    'Gram-':       [0,0,1,0,0,0,0],
    'Antifungal':  [0,0,0,1,0,0,0],
    'Anticancer':  [0,0,0,0,0,0,1],
}

for name, cond in tqdm(conditions_for_analysis.items(), desc="Generating per condition"):
    seqs = [generate(model, cond) for _ in trange(n_gen, desc=name, leave=False)]
    seqs = [s for s in seqs if len(s) > 0]  # filter empty
    gen_data[name] = compute_properties(seqs)
    print(f"{name}: generated {len(seqs)} valid sequences")

# Also compute properties for real training data by condition
real_data = {}
for name, col_idx in [('Gram+', 1), ('Gram-', 2), ('Antifungal', 3), ('Anticancer', 6)]:
    mask = df[condition_cols[col_idx]] == 1
    seqs = df.loc[mask, seq_col].tolist()
    real_data[name] = compute_properties(seqs)
    print(f"Real {name}: {len(seqs)} sequences")

In [ ]:
# =====================================================================
# Physicochemical Property Plots
# =====================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

properties = ['net_charge', 'hydrophobicity', 'hydrophobic_moment', 'length']
titles = ['Net Charge (pH 7)', 'Hydrophobicity (Kyte-Doolittle)',
          'Hydrophobic Moment (Amphipathicity)', 'Sequence Length']

for ax, prop, title in zip(axes.flat, properties, titles):
    for name in conditions_for_analysis:
        # Generated
        vals = gen_data[name][prop].values
        ax.hist(vals, bins=30, alpha=0.4, label=f'Gen {name}', density=True)
    # Overlay real data for one condition as reference
    for name in ['Gram+']:
        vals = real_data[name][prop].values
        ax.hist(vals, bins=30, alpha=0.3, label=f'Real {name}',
                density=True, color='black', histtype='step', linewidth=2)
    ax.set_title(title)
    ax.set_xlabel(prop)
    ax.legend(fontsize=8)

plt.suptitle('Physicochemical Properties: Generated vs Real AMPs', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Charge vs Hydrophobicity scatter (colored by condition)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Generated
ax = axes[0]
for name, props_df in gen_data.items():
    ax.scatter(props_df['net_charge'], props_df['hydrophobicity'],
              alpha=0.4, s=15, label=name)
ax.set(xlabel='Net Charge', ylabel='Mean Hydrophobicity',
       title='Generated Peptides: Charge vs Hydrophobicity')
ax.legend()
ax.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax.axvline(0, color='gray', linestyle='--', alpha=0.3)

# Real
ax = axes[1]
for name, props_df in real_data.items():
    ax.scatter(props_df['net_charge'], props_df['hydrophobicity'],
              alpha=0.4, s=15, label=name)
ax.set(xlabel='Net Charge', ylabel='Mean Hydrophobicity',
       title='Real Peptides: Charge vs Hydrophobicity')
ax.legend()
ax.axhline(0, color='gray', linestyle='--', alpha=0.3)
ax.axvline(0, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Property statistics table
print(f"{'':20} {'Charge':>10} {'Hydro':>10} {'HydroMom':>10} {'Length':>10}")
print("=" * 65)
for label, data_dict in [('Generated', gen_data), ('Real', real_data)]:
    for name, props_df in data_dict.items():
        print(f"{label+' '+name:20} "
              f"{props_df['net_charge'].mean():>10.2f} "
              f"{props_df['hydrophobicity'].mean():>10.2f} "
              f"{props_df['hydrophobic_moment'].mean():>10.2f} "
              f"{props_df['length'].mean():>10.1f}")

## 8. Condition Strength Interpolation Experiment

Inspired by Dean & Walper's active-to-scrambled interpolation:
we vary the condition strength alpha from 1 (full condition) to 0
(no condition) and analyze how physicochemical properties change.

In [ ]:
# Generate sequences at different condition strengths
alpha_values = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
cond_target = [0, 1, 0, 0, 0, 0, 0]  # Gram+
n_per_alpha = 100

alpha_results = {}
for alpha in tqdm(alpha_values, desc="Alpha sweep"):
    seqs = [generate(model, cond_target, alpha=alpha) for _ in range(n_per_alpha)]
    seqs = [s for s in seqs if len(s) > 0]
    alpha_results[alpha] = compute_properties(seqs)
    print(f"alpha={alpha:.1f}: {len(seqs)} sequences, "
          f"mean charge={alpha_results[alpha]['net_charge'].mean():.2f}, "
          f"mean hydro={alpha_results[alpha]['hydrophobicity'].mean():.2f}")

In [ ]:
# Plot property trends vs condition strength
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, prop, title in zip(axes, properties, titles):
    means = [alpha_results[a][prop].mean() for a in alpha_values]
    stds = [alpha_results[a][prop].std() for a in alpha_values]
    ax.errorbar(alpha_values, means, yerr=stds, marker='o', capsize=3)
    ax.set(xlabel='Condition Strength (alpha)', ylabel=prop, title=title)
    ax.grid(True, alpha=0.3)

plt.suptitle('Physicochemical Properties vs Condition Strength (Gram+)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 9. Example Usage

Practical examples of generating and analyzing peptides with different conditions.

In [ ]:
# Example 1: Generate peptides for specific conditions
print("=" * 70)
print("Example 1: Single-condition generation")
print("=" * 70)
for name, cond in conditions_for_analysis.items():
    print(f"\n{name}:")
    for i in range(3):
        seq = generate(model, cond, temperature=0.8)
        print(f"  {seq} (len={len(seq)}, charge={net_charge(seq):.1f}, "
              f"hydro={hydrophobicity(seq):.2f})")

In [ ]:
# Example 2: Multi-hot conditions (peptide with multiple activities)
print("=" * 70)
print("Example 2: Multi-hot condition generation")
print("=" * 70)

multi_conditions = {
    'Antibacterial + Gram+ + Gram-': [1, 1, 1, 0, 0, 0, 0],
    'Gram+ + Antifungal':            [0, 1, 0, 1, 0, 0, 0],
    'Broad spectrum (all)':           [1, 1, 1, 1, 1, 1, 1],
}

for name, cond in multi_conditions.items():
    print(f"\n{name} {cond}:")
    for i in range(3):
        seq = generate(model, cond, temperature=0.8)
        print(f"  {seq} (len={len(seq)}, charge={net_charge(seq):.1f})")

In [ ]:
# Example 3: Latent space interpolation between two real peptides
print("=" * 70)
print("Example 3: Latent interpolation between two encoded peptides")
print("=" * 70)

# Encode two real peptides
seq1 = df.iloc[0][seq_col]  # First peptide
seq2 = df.iloc[100][seq_col]  # Another peptide
cond1 = conditions[0].tolist()
cond2 = conditions[100].tolist()

# Encode to latent space
tok1 = torch.LongTensor([tokenize(seq1, char2idx, max_len)]).to(device)
len1 = torch.LongTensor([min(len(seq1)+2, max_len)]).to(device)
cond1_t = torch.FloatTensor([cond1]).to(device)

tok2 = torch.LongTensor([tokenize(seq2, char2idx, max_len)]).to(device)
len2 = torch.LongTensor([min(len(seq2)+2, max_len)]).to(device)
cond2_t = torch.FloatTensor([cond2]).to(device)

model.eval()
with torch.no_grad():
    z1, _ = model.encoder(tok1, len1, cond1_t)
    z2, _ = model.encoder(tok2, len2, cond2_t)

print(f"Peptide 1: {seq1[:40]}... (cond={cond1})")
print(f"Peptide 2: {seq2[:40]}... (cond={cond2})")
print()

# Interpolate
results = interpolate_latent(model, z1, z2, cond1, n_steps=6, temperature=0.8)
for alpha, seq in results:
    print(f"  alpha={alpha:.2f}: {seq} (len={len(seq)}, charge={net_charge(seq):.1f})")

In [ ]:
# Example 4: Temperature sweep
print("=" * 70)
print("Example 4: Effect of temperature on generation diversity")
print("=" * 70)
cond_gp = [0, 1, 0, 0, 0, 0, 0]
for temp in [0.5, 0.8, 1.0, 1.2]:
    seqs = set(generate(model, cond_gp, temperature=temp) for _ in range(50))
    avg_len = np.mean([len(s) for s in seqs if len(s) > 0])
    print(f"  T={temp:.1f}: {len(seqs)} unique / 50, avg length={avg_len:.1f}")

In [ ]:
# Save models and vocabulary
with open('vocab.pkl', 'wb') as f:
    pickle.dump({
        'char2idx': char2idx, 'idx2char': idx2char,
        'max_len': max_len, 'vocab_list': vocab_list,
        'condition_cols': condition_cols
    }, f)

print("Saved: best_cvae.pt, parametric_tsne.pt, vocab.pkl")
print("Done!")